# CUPE Local 3902 Unit 1 — Historical Job Postings Analysis

Dataset scraped from [unit1.hrandequity.utoronto.ca](https://unit1.hrandequity.utoronto.ca) — all CUPE 3902 Unit 1 job postings (Teaching Assistants, Course Instructors, Invigilators, etc.) at the University of Toronto from 2021 onwards.

**46,735 postings** • **3 campuses** (St. George, UTM, UTSC) • **300 departments**

---

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('seaborn-darkgrid')

## Load & Clean Data

In [ ]:
conn = sqlite3.connect('/kaggle/input/uoft-cupe-unit1-job-postings/history.db')
df = pd.read_sql_query('SELECT * FROM postings', conn)
conn.close()

# Parse dates
for col in ['posting_date', 'closing_date', 'expiry_date',
            'appointment_startdate', 'appointment_enddate', 'created_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

# Filter to realistic date range (some rows have malformed dates)
df = df[df['posting_date'].dt.year.between(2020, 2027)]

# Derived columns
df['posting_year']     = df['posting_date'].dt.year
df['posting_month']    = df['posting_date'].dt.month
df['posting_month_nm'] = df['posting_date'].dt.strftime('%b')
df['posting_dow']      = df['posting_date'].dt.dayofweek        # 0=Mon
df['posting_dow_nm']   = df['posting_date'].dt.strftime('%a')
df['lead_days']        = (df['closing_date'] - df['posting_date']).dt.days
df['positions_num']    = pd.to_numeric(df['positions'], errors='coerce')
df['appt_hours']       = pd.to_numeric(
    df['appointment_size'].str.extract(r'(\d+)')[0], errors='coerce'
)

MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
DOW_ORDER   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

print(f'Loaded {len(df):,} postings')
df.head(3)

In [ ]:
df.info()

## Overview Statistics

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total postings',
        'Date range',
        'Unique campuses',
        'Unique departments',
        'Unique courses',
        'Emergency postings',
        'Median lead time (post → close)',
        'Median positions per posting',
        'Median appointment hours',
    ],
    'Value': [
        f"{len(df):,}",
        f"{int(df['posting_year'].min())} – {int(df['posting_year'].max())}",
        df['campus_name'].nunique(),
        df['department_name'].nunique(),
        df['course_id'].nunique(),
        f"{df['emergency'].sum():,} ({df['emergency'].mean()*100:.1f}%)",
        f"{df['lead_days'].median():.0f} days",
        f"{df['positions_num'].median():.0f}",
        f"{df['appt_hours'].median():.0f} h",
    ]
})
summary.set_index('Metric')

## 1 · Postings Over Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly area chart
monthly = (
    df.dropna(subset=['posting_date'])
      .groupby(df['posting_date'].dt.tz_localize(None).dt.to_period('M'))
      .size()
      .sort_index()
)
monthly.plot(ax=axes[0], kind='area', color='#4c72b0', alpha=0.7, linewidth=1.2)
axes[0].set_title('Postings per Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('Postings')

# Annual bar chart
annual = df.dropna(subset=['posting_year']).groupby('posting_year').size()
annual.plot(kind='bar', ax=axes[1], color='#4c72b0', edgecolor='white')
axes[1].set_title('Postings per Year')
axes[1].set_xlabel('Year')
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('Volume of Job Postings Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2 · Seasonal & Weekly Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Average postings by month across all years
monthly_avg = (
    df.dropna(subset=['posting_month_nm'])
      .groupby(['posting_year', 'posting_month_nm'])
      .size().reset_index(name='count')
      .groupby('posting_month_nm')['count'].mean()
      .reindex(MONTH_ORDER)
)
monthly_avg.plot(kind='bar', ax=axes[0], color='#dd8452', edgecolor='white')
axes[0].set_title('Avg Postings by Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('Avg postings')
axes[0].tick_params(axis='x', rotation=0)

# Day of week
dow_counts = df['posting_dow_nm'].value_counts().reindex(DOW_ORDER).fillna(0)
dow_counts.plot(kind='bar', ax=axes[1], color='#8172b3', edgecolor='white')
axes[1].set_title('Postings by Day of Week')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('When Are Jobs Posted?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Season breakdown
season_map = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
               6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'}
df['season'] = df['posting_month'].map(season_map)
print('\nPostings by Season:')
print(df['season'].value_counts().reindex(['Fall','Winter','Spring','Summer']))

## 3 · Campus & Position Type Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Campus pie
campus_counts = df['campus_name'].value_counts()
axes[0].pie(campus_counts, labels=campus_counts.index, autopct='%1.1f%%',
            startangle=140, pctdistance=0.82,
            colors=['#4c72b0','#dd8452','#55a868'])
axes[0].set_title('Postings by Campus')

# Position type
ptype = df['position_type_name'].value_counts()
ptype.plot(kind='bar', ax=axes[1], color='#8172b3', edgecolor='white')
axes[1].set_title('Postings by Position Type')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 4 · Top Departments & Courses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_depts = df['department_name'].value_counts().head(20)
top_depts[::-1].plot(kind='barh', ax=axes[0], color='#4c72b0', edgecolor='white')
axes[0].set_title('Top 20 Departments')
axes[0].set_xlabel('Postings')

top_courses = df['course_id'].dropna().value_counts().head(20)
top_courses[::-1].plot(kind='barh', ax=axes[1], color='#55a868', edgecolor='white')
axes[1].set_title('Top 20 Most-Posted Courses')
axes[1].set_xlabel('Times Posted')

plt.tight_layout()
plt.show()

## 5 · St. George Engineering & CS — Posting Timing

When do engineering and CS TA jobs at the St. George campus typically get posted?

In [ ]:
ENG_DEPT = r'engineer|comput|electrical|mechanic|chem|civil|material|ece|csc|aps|aer|phy|mie|esc'
ENG_CODE = r'^(APS|ECE|CSC|ESC|MIE|AER|CHM|PHY|MSE|MEC|BME|CHE|CIV|IND|JRE|ROB|MAT|STA)'

sg_eng = df[
    (df['campus_name'].str.lower() == 'st. george') &
    (
        df['department_name'].str.contains(ENG_DEPT, case=False, na=False) |
        df['course_id'].str.match(ENG_CODE, case=False, na=False)
    )
]
print(f'St. George Engineering/CS postings: {len(sg_eng):,}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Monthly
m = sg_eng['posting_month_nm'].value_counts().reindex(MONTH_ORDER).fillna(0)
m.plot(kind='bar', ax=axes[0], color='#55a868', edgecolor='white')
axes[0].set_title('By Month')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylabel('Postings')
axes[0].axvline(m.values.argmax() - 0.5, color='red', linestyle='--', alpha=0.5)
print(f'Peak month: {m.idxmax()} ({int(m.max())} postings)')

# Day of week
d = sg_eng['posting_dow_nm'].value_counts().reindex(DOW_ORDER).fillna(0)
d.plot(kind='bar', ax=axes[1], color='#c44e52', edgecolor='white')
axes[1].set_title('By Day of Week')
axes[1].tick_params(axis='x', rotation=0)
print(f'Peak day: {d.idxmax()}')

# Yearly trend
y = sg_eng.dropna(subset=['posting_year']).groupby('posting_year').size()
y.plot(kind='bar', ax=axes[2], color='#4c72b0', edgecolor='white')
axes[2].set_title('By Year')
axes[2].tick_params(axis='x', rotation=0)

fig.suptitle('St. George Engineering/CS TA Posting Timing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 · Emergency Posting Rate Over Time

In [ ]:
yearly = df.dropna(subset=['posting_year']).groupby('posting_year').agg(
    total=('emergency', 'count'),
    emerg=('emergency', 'sum')
)
yearly['rate'] = yearly['emerg'] / yearly['total'] * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(yearly.index, yearly['emerg'], label='Emergency', color='#c44e52', edgecolor='white')
bars2 = ax.bar(yearly.index, yearly['total'] - yearly['emerg'],
               bottom=yearly['emerg'], label='Regular', color='#4c72b0', edgecolor='white')
ax2 = ax.twinx()
ax2.plot(yearly.index, yearly['rate'], color='black', marker='o', linewidth=2, label='Emergency %')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_ylabel('Emergency %')
ax.set_title('Emergency vs Regular Postings per Year', fontweight='bold')
ax.set_ylabel('Postings')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('\nEmergency rate by department (top 10):')
dept_emerg = df.groupby('department_name').agg(total=('emergency','count'), emerg=('emergency','sum'))
dept_emerg = dept_emerg[dept_emerg['total'] >= 20]  # min 20 postings
dept_emerg['rate'] = dept_emerg['emerg'] / dept_emerg['total'] * 100
print(dept_emerg.nlargest(10, 'rate')[['total','emerg','rate']].rename(columns={'rate':'emerg_%'}).round(1))

## 7 · Lead Time (Posting → Closing Date)

In [ ]:
lead = df['lead_days'].dropna()
lead_ok = lead[(lead >= 0) & (lead <= 90)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lead_ok, bins=45, color='#4c72b0', edgecolor='white', alpha=0.85)
axes[0].axvline(lead_ok.median(), color='#c44e52', linestyle='--', linewidth=2,
                label=f'Median: {lead_ok.median():.0f} days')
axes[0].set_title('Distribution of Lead Time (all postings)')
axes[0].set_xlabel('Days (posting → closing)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Lead time by position type
lead_by_type = df[df['lead_days'].between(0, 90)].groupby('position_type_name')['lead_days'].median().sort_values()
lead_by_type.plot(kind='barh', ax=axes[1], color='#dd8452', edgecolor='white')
axes[1].set_title('Median Lead Time by Position Type')
axes[1].set_xlabel('Days')

plt.tight_layout()
plt.show()

print(f"Lead time stats (0–90 day range):")
print(lead_ok.describe().round(1))

## 8 · Appointment Hours Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hrs = df['appt_hours'].dropna()
hrs_ok = hrs[(hrs >= 1) & (hrs <= 500)]
axes[0].hist(hrs_ok, bins=60, color='#dd8452', edgecolor='white', alpha=0.85)
axes[0].axvline(hrs_ok.median(), color='#4c72b0', linestyle='--', linewidth=2,
                label=f'Median: {hrs_ok.median():.0f} h')
axes[0].set_title('Distribution of Appointment Size (hours)')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Count')
axes[0].legend()

# Median hours by campus
hrs_campus = df[df['appt_hours'].between(1, 500)].groupby('campus_name')['appt_hours'].median()
hrs_campus.plot(kind='bar', ax=axes[1], color='#55a868', edgecolor='white')
axes[1].set_title('Median Appointment Hours by Campus')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 9 · Course Enrolment vs Number of TAs

In [ ]:
scatter_df = df.copy()
scatter_df['enrolment_num'] = pd.to_numeric(scatter_df['course_enrolment'], errors='coerce')
scatter_df = scatter_df.dropna(subset=['enrolment_num', 'positions_num'])
scatter_df = scatter_df[
    scatter_df['enrolment_num'].between(1, 3000) &
    scatter_df['positions_num'].between(1, 50)
]

fig, ax = plt.subplots(figsize=(10, 5))
colors = {'St. George': '#4c72b0', 'UTM': '#dd8452', 'UTSC': '#55a868'}
for campus, grp in scatter_df.groupby('campus_name'):
    ax.scatter(grp['enrolment_num'], grp['positions_num'],
               alpha=0.25, s=15, label=campus,
               color=colors.get(campus, 'grey'))

# Trend line
z = np.polyfit(scatter_df['enrolment_num'], scatter_df['positions_num'], 1)
p = np.poly1d(z)
xs = np.linspace(0, 3000, 200)
ax.plot(xs, p(xs), 'r--', linewidth=1.5, label=f'Trend (slope={z[0]:.4f})')

ax.set_title('Course Enrolment vs. Number of TA Positions', fontweight='bold')
ax.set_xlabel('Course Enrolment')
ax.set_ylabel('Number of Positions')
ax.legend()
plt.tight_layout()
plt.show()
print(f'\nCorrelation: {scatter_df["enrolment_num"].corr(scatter_df["positions_num"]):.3f}')

## 10 · Heatmap — Posting Activity by Month × Day of Week

In [ ]:
heatmap_df = df.dropna(subset=['posting_month_nm', 'posting_dow_nm']).copy()
pivot = heatmap_df.groupby(['posting_dow_nm', 'posting_month_nm']).size().unstack(fill_value=0)
pivot = pivot.reindex(index=DOW_ORDER, columns=MONTH_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_ORDER)
ax.set_yticks(range(7))
ax.set_yticklabels(DOW_ORDER)
plt.colorbar(im, ax=ax, label='Postings')
ax.set_title('Posting Activity Heatmap — Month × Day of Week', fontweight='bold')
plt.tight_layout()
plt.show()

## 11 · Explore the Raw Data

In [ ]:
# Filter to any subset you want
# Example: St. George TA postings closing in 2024
sample = df[
    (df['campus_name'] == 'St. George') &
    (df['position_type_short'] == 'TA') &
    (df['closing_date'].dt.year == 2024)
].sort_values('posting_date', ascending=False)

print(f'{len(sample):,} rows')
sample[['id','course_id','job_title','department_name','positions','appointment_size',
        'posting_date','closing_date','emergency']].head(10)